# Example: Single spin Part 2: Gradient descent gate optimization

First, we make the necessary imports.

In [ ]:
import matplotlib.pyplot as plt

import numpy as np

from cthree.OptimisationMap import OptimisationMap
from cthree.Quantity import Quantity
from cthree.measurement.UnitaryFidelity import UnitaryFidelity
from cthree.model.GeneratorDrive import GeneratorDrive
from cthree.model.Qubit import Qubit
from cthree.propagation.ScipyExpmGOAT import ScipyExpmGOAT
from cthree.optimisers.ScipyOptimiserGradient import ScipyOptimiserGradient

from cthree.model.ClosedModel import ClosedModel

from cthree.signal.SimpleGenerator import CosGenerator
from cthree.signal.Device import CosTone

For signal generation, we define a simple cosine shaped tone generator $A \cos(\omega t)$

In [ ]:
tone = CosTone()
gen = CosGenerator(devices=[tone])

We can inspect the pre-defined parameters with

In [ ]:
params = tone.getParameters()
params

in this case amplitude $A$ and frequency $\omega$ and phase $\phi$.

Next, we setup the qubit system we want to control. We set the qubit frequency $\omega_q$ to be 4.8 GHz and define the Hamiltonian as $$H(t)=H_\text{drift}+H_c(t)= \frac{\omega_q}{2} \sigma_z + \Omega(t)\sigma_x$$, where $\Omega(t)=A\cos(\omega t+\phi)$ will be supplied by the generator.

In [ ]:
FREQ = 4.327884e9 * 2 * np.pi

drive = GeneratorDrive(gen, isLongitudinal=False)
controlled_qubit = Qubit(
    frequency=Quantity(
        FREQ,
        min_value=FREQ / 4,
        max_value=FREQ,
        unit="Hz",
        name="Qubit frequency",
    ),
    drives=[drive],
)

model = ClosedModel(controlled_qubit)

Textbook values for implementing an $X$ rotation on this system at a time $T$ would be $\omega=\omega_q$ and $A=\pi/T$. We use some offset from these values as initial guess to demonstrate the optimization procedure.

In [ ]:
t_final = 10e-9
params[0].setValue(0.5 * np.pi / t_final)
params[1].setValue(1.01 * FREQ)

We select a propagation method, piecewise constant exponentation, and configure $\sigma_x$ as a target gate. Also we initialize the full basis at time 0 with $\mathcal{I}_2$

In [ ]:
prop = ScipyExpmGOAT(model, res=100e9)

X = np.array([[0.0, 1], [1, 0.0]])
Y = np.array([[0.0, -1j], [1j, 0.0]])
Z = np.array([[1, 0], [0.0, -1]])

prop.setInitialState(np.identity(2))
gateFid = UnitaryFidelity(
    propagation=prop,
    gate=X,
    times=np.array([0.0, t_final]),
)

In [ ]:
def plotStates():
    """Plot the states."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generateSignal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax


plotStates()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
gateFid.measure()

We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone and optimise just amplitude and frequency, as in the state transfer example.

In [ ]:
optmap = OptimisationMap()
optmap.add(tone, [params[0], params[1]])
opt = ScipyOptimiserGradient(gateFid, optimisables=optmap)

In [ ]:
opt.optimise()

In [ ]:
plotStates()

Here, we end up with an unsatisfactory result. For optimising gate fidelities, looking at state populations does not give enough information to identify the problem.

In [ ]:
def expecationValue(Op, states):
    """Get the expectation value across states."""
    ex = []
    for state in states:
        ex.append(np.real(state.conj() @ Op @ state.T))
    return ex

In [ ]:
def plotPauli():
    """Plot the Pauli operators."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generateSignal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, expecationValue(X, states[:, :, 0]))
    ax[1].plot(ts / 1e-9, expecationValue(Y, states[:, :, 0]))
    ax[1].plot(ts / 1e-9, expecationValue(Z, states[:, :, 0]))
    ax[1].set_ylabel(r"Expectation value $\langle\hat\sigma_i\rangle$")
    ax[-1].set_xlabel("Time [ns]")
    ax[1].legend(["X", "Y", "Z"])
    return fig, ax


plotPauli()

Instead, we look at the expecation values of the three Pauli operators and observe that the qubit is rotating at its eigenfrequency along the Z-axis. We can mitigate this problem by allowing the rotation axis of our drive to shift and inclide the phase parameter in the optimisation.

In [ ]:
optmap = OptimisationMap()
optmap.add(tone, params)
opt = ScipyOptimiserGradient(gateFid, optimisables=optmap)

In [ ]:
params[0].setValue(0.5 * np.pi / t_final)
params[1].setValue(1.01 * FREQ)
opt.optimise()

In [ ]:
plotPauli()

In [ ]:
params